In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append("..")
from dataloader_comma import CommaDataset
from expand_road_events import *
from collections import Counter
import imageio
from model import VTN
import matplotlib.pyplot as plt 
from PIL import Image
import glob
plt.rcParams.update({'font.size': 26}) 
import os
from utils import * 
import re
from vis_utils import * 
from tqdm import tqdm
import warnings 
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

Checkpoint Root

In [ ]:
from torch.utils.data import DataLoader
from dataloader_comma import CommaDataset

# Create the dataset
dataset_comma = CommaDataset(
    dataset_type="test",  # o "val" o "train" depends on what we want plot
    use_transform=False,
    multitask="distance",  
    ground_truth="desired", # Change to True to have all the parameters you use in the plot
    dataset_path="/kaggle/input/final-hdf5-files",
    dataset_fraction=1.0
)

print(f"Dataset created with {len(dataset_comma)} samples")

# Creating the Dataloader
dataloader_comma = DataLoader(
    dataset_comma,
    batch_size=1,  # Keep 1 for plotting
    shuffle=False,  # Don't shuffle for plots
    num_workers=0,  # 0 for easier debugging
    # collate_fn=None  # Use the default collate_fn, NOT the custom one you showed
)
model = VTN(multitask='distance', backbone='none', concept_features= True, device = 'cuda:1', return_concepts=True, concept_source = 'retinanet')

print("DataLoader created successfully")

In [ ]:
ckpt_root = f"/kaggle/working/ckpts_final_comma_distance_none_True_1_clip"
#find the latest version -G.R.
versions = glob.glob(os.path.join(ckpt_root, "lightning_logs", "version_*"))
if not versions:
    raise FileNotFoundError("None found")
latest_version = max(versions, key=os.path.getmtime)

# Find checkpoints in the latest version -G.R.
ckpt_files = glob.glob(os.path.join(latest_version, "checkpoints", "*.ckpt"))
if not ckpt_files:
    raise FileNotFoundError("None found.")
checkpoint_path = max(ckpt_files, key=os.path.getmtime)

print(f"Using checkpoint: {checkpoint_path}")

ckpt = torch.load(checkpoint_path)
state_dict = ckpt['state_dict']
state_dict = get_regular_ckpt_from_lightning_checkpoint(state_dict)
#added this to load the model correctly -G.R.
model.load_state_dict(state_dict)
print('done')

In [ ]:
model.eval()
model = model.to(gpu)

In [ ]:
def split_string(string):
    words = string.replace("a photo of driving on a highway with", "").replace("a photo of", "").replace("driving on a highway", "").replace("past", "").replace("a street with", "").split()  # Split the string into a list of words
    result = []
    current_line = ""
    
    for word in words:
        if len(current_line) + len(word) <= 33:
            current_line += word + " "
        else:
            result.append(current_line.strip())
            current_line = word + " "
    
    if current_line:
        result.append(current_line.strip())
    
    return "\n".join(result)

In [ ]:
import os

image_dir = "/kaggle/working/results_images"
gif_dir = "/kaggle/working/results_images/att"

os.makedirs(image_dir, exist_ok=True)
os.makedirs(gif_dir, exist_ok=True)


In [ ]:
sys.path.append("..")
from expand_road_events import expand_label

In [ ]:
from utils import get_scenarios

scenarios, scenarios_tokens = get_scenarios("retinanet")   #or clip

scenarios = list(scenarios)

In [ ]:
for j, batch in enumerate(dataloader_comma):
    image_array,  vego, angle, distance, g, s, l, seq_key = batch #g= gaspressed, s= brakepressed, l= cruiseenabled
        
    img = image_array
    max_len = 240
    # subito dopo il batch unpacking
    img, angle, distance, vego = img.to(gpu), angle.to(gpu), distance.to(gpu), vego.to(gpu)
    (logits, attns), concepts = model(img, angle, distance, vego, seq_key)
    
    top5_indices = torch.tensor(concepts.squeeze()).topk(10).indices
    s = img.shape
    angle, distance, vego, logits, concepts = angle.to("cpu"), distance.to("cpu"), vego.to("cpu"), logits.detach().cpu().to("cpu"), concepts.detach().cpu().to("cpu")
    
    f = []
    inter = []
    for i, elem0 in enumerate(top5_indices):
        inter = []
        for elem in top5_indices[max(i-20, 0):min(i+20,len(top5_indices))]:
            l = elem.cpu().numpy().tolist()
            if 131 in l:
                l.remove(131)
            if 55 in l:
                l.remove(55)
            inter.extend(l)
        count_dict = Counter(inter)
        # Get the top 5 most occurring numbers
        top_5 = count_dict.most_common(3)
        intermediate = []
        for a in top_5: 
            intermediate.append(scenarios[a[0]])
        f.append(intermediate)

    fig, axes = plt.subplots(1, 1, figsize=(12, 16))#,gridspec_kw= {'height_ratios': [20, 1]})

    plt_idx = 0
    for i, image in tqdm(enumerate(img[0][10:120])):
    
        image_frame = (image).cpu().permute(1, 2, 0)#unorm(image).cpu().permute(1, 2, 0)

        # Display the image frame
        axes.imshow((np.array(image_frame) * 255).astype(np.uint8))
        
        #title = '\n'.join([split_string("\u2022 " + h) for h in f[i]]) # ORIGINAL
        lines = [split_string(h) for h in f[i]]               # senza bullet
        expanded = [expand_label(line) for line in lines]     # espandi le frasi
        title = '\n'.join("\u2022 " + e for e in expanded)    # riaggiungi il bullet
                
        axes.set_title(title)
        axes.set_aspect('equal')
        axes.set_xticks([])
        axes.set_yticks([])

        # Remove borders
        axes.spines['top'].set_visible(False)
        axes.spines['bottom'].set_visible(False)
        axes.spines['left'].set_visible(False)
        axes.spines['right'].set_visible(False)

        plt.savefig(f"/kaggle/working/results_images/{i}.png") #Modified to save images in Kaggle environment

    
    image_directory = '/kaggle/working/results_images/v2'

    # Set the output GIF file path
    output_path = lambda x: f'/kaggle/working/results_images/att/attention_comma_{j}.{x}'

    # Set the duration (in milliseconds) for each frame in the GIF
    frame_duration = 700

    # Get a sorted list of image files in the directory
    image_files = sorted(glob.glob(f'{image_directory}/*.png'), key=extract_number)  # Adjust the file extension if necessary
    
    # Create a list to store the frames of the GIF
    frames = []

    # Iterate over each image file
    for image_file in image_files:
        # Open the image file
        image = Image.open(image_file)

        # Add the image to the list of frames
        frames.append(image)

    # Save the frames as a GIF
    #frames[0].save(output_path("gif"), format='GIF', append_images=frames[1:], save_all=True,
    #            duration=frame_duration, loop=0)
    #imageio.mimsave(output_path("mp4"), frames, fps=4)
    
    if j > 5: break

In [ ]:
# ============================
# SINGLE LAYER ATTENTION VISUALIZATION WITH VIDEO SELECTION
# ============================
import os, gc, glob
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')   # non-interactive backend, avoids renderer issues
import matplotlib.pyplot as plt
from matplotlib import patches
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.signal import find_peaks
from collections import Counter
from PIL import Image
import imageio
from tqdm import tqdm

# ---------------- Parameters (modify as needed) ----------------
outdir = "/kaggle/working/results_single_layer_noise_retinanet"
os.makedirs(outdir, exist_ok=True)

chunk_size = 8            # number of columns per figure
fig_w_per_col = 4.0
fig_h = 6.0
dpi_save = 200
make_gif = True
normalize_speed = False   # if True normalizes each speed_graph (useful for very flat curves)
save_model_checkpoint = False
checkpoint_path = "/kaggle/input/retinanet-ckpt-seed42/ckpts_final_comma_distance_none_True_1_retinanet"
verbose = True

# New parameters for marker/zoom
#zoom_window = 20          # +/- steps shown around current frame (adjust here)
marker_size = 48
marker_color = "red"
annotate_values = True    # show numeric value next to marker

# Video selection parameter - specify the video you want to process
TARGET_VIDEO = "b0c9d2329ad1606b_2018-07-30--13-03-07_16"  # Change this to select specific video
PROCESS_ALL_VIDEOS = False  # Set to True to process all videos, False to process only TARGET_VIDEO
# --------------------------------------------------------------

# helper safe save (draw before save to avoid NoneType renderer)
def safe_plot_and_save(fig, png_path, dpi=dpi_save):
    try:
        fig.canvas.draw()               # build renderer
        fig.savefig(png_path, dpi=dpi, bbox_inches='tight')
    except Exception as e:
        try:
            # fallback: try without tight bbox/inset
            print(f"[WARN] first save failed: {e}. Trying fallback.")
            fig.savefig(png_path, dpi=dpi)
        except Exception as e2:
            print(f"[ERROR] fallback save failed: {e2}")
            raise e2
    finally:
        plt.close(fig)

# simple string split for titles
import re
import textwrap

def split_string(s: str, max_len: int = 33) -> str:
    """Wrap a label string into lines of at most `max_len` chars without breaking words.
       Performs the same initial replacements you had.
    """
    if s is None:
        return ""
    s = s.strip()
    if not s:
        return ""
    if len(s) <= max_len:
        return s
    wrapped = textwrap.wrap(s, width=max_len, break_long_words=False, break_on_hyphens=False)
    return "\n".join(wrapped)

# get device from model
device = next(model.parameters()).device
print(f"[INFO] model device: {device}")

saved_pngs = []

# ---------- loop over batches ----------
for j, batch in enumerate(dataloader_comma):
    # dataloader returns 8 elements: images, vego, angle, distance, gas, brake, cruise, seq_key
    image_array, vego, angle, distance, gas, brake, cruise, seq_key = batch

    # Check if we should process this video
    current_video_id = seq_key[0] if isinstance(seq_key, (list, tuple)) else str(seq_key)
    
    if not PROCESS_ALL_VIDEOS:
        if TARGET_VIDEO not in current_video_id:
            print(f"[SKIP] Video {current_video_id} - looking for {TARGET_VIDEO}")
            continue
        else:
            print(f"[PROCESSING] Found target video: {current_video_id}")
    else:
        print(f"[PROCESSING] Video: {current_video_id}")

    # move input to model device and forward
    img = image_array.to(device)
    angle_dev = angle.to(device); distance_dev = distance.to(device); vego_dev = vego.to(device)

    # forward pass (try/except if model can fail)
    with torch.no_grad():
        (logits, attns), concepts = model(img, angle_dev, distance_dev, vego_dev, seq_key)

    print(f"[BATCH {j}] Video: {current_video_id}")
    print(f"[BATCH {j}] Number of attention layers: {len(attns) if isinstance(attns, (list, tuple)) else 1}")

    # move to CPU for plotting (img_cpu shape: [B, seq_len, C, H, W])
    img_cpu = img.detach().cpu()

    # concepts to CPU
    concepts_cpu = concepts.detach().cpu()

    # attns to CPU (handle list/tuple or None)
    if attns is None:
        attns_cpu = []
    elif isinstance(attns, (list, tuple)):
        attns_cpu = [a.detach().cpu() for a in attns]
    else:
        attns_cpu = [attns.detach().cpu()]

    # function that produces speed graph from an attention layer (catch errors)
    def compute_speed_from_att(atten_cpu):
        try:
            seq_len_local = atten_cpu.shape[2]
            alignment_array = get_aligned_attention(atten_cpu.squeeze().cpu(), seq_len_local)
            g = alignment_array.sum(axis=0)[8:-8]
            return moving_average(g, 10)
        except Exception as e:
            if verbose: print(f"[WARN] compute_speed failed: {e}")
            return np.array([])

    # calculate speed_graph only for the LAST layer (most semantically rich)
    if len(attns_cpu) > 0:
        last_layer_att = attns_cpu[-1]  # take last layer
        speed_graph = compute_speed_from_att(last_layer_att[:,:,0:concepts_cpu.shape[1]])
        print(f"[BATCH {j}] Using last layer (index {len(attns_cpu)-1}) for attention visualization")
    else:
        speed_graph = np.array([])
        print(f"[BATCH {j}] No attention data available")

    # minimal diagnostics
    if verbose and speed_graph.size > 0:
        print(f"[BATCH {j}] attention signal - min/max/mean: {speed_graph.min():.6f}, {speed_graph.max():.6f}, {speed_graph.mean():.6f}")
        print(f"[BATCH {j}] attention signal length: {len(speed_graph)}")
    elif verbose:
        print(f"[BATCH {j}] attention signal empty")

    # build titles (top concepts) - as in original notebook
    topk = torch.tensor(concepts_cpu.squeeze()).topk(5).indices
    f = []
    for i, _ in enumerate(topk):
        inter = []
        for elem in topk[max(i-5, 0):min(i+5, len(topk))]:
            l = elem.cpu().numpy().tolist()
            for bad in (131, 55):
                if bad in l:
                    l.remove(bad)
            inter.extend(l)
        cdict = Counter(inter)
        top_2 = cdict.most_common(2)
        f.append([expand_label(scenarios[a[0]]) for a in top_2])
            
    # optional: build intervention mask for highlighting (uses gas/brake/cruise)
    try:
        intervention_mask = (np.array(gas).astype(bool) | np.array(brake).astype(bool) | (~np.array(cruise).astype(bool)))
    except Exception:
        intervention_mask = None

    # indices of frames to plot
    seq_len = img_cpu.shape[1]
    frames_indices = list(range(10, seq_len - 10))
    if len(frames_indices) == 0:
        print("[WARN] no valid frames (range 10:-10 empty). Skip batch.")
        # cleanup and continue
        del img, angle_dev, distance_dev, vego_dev, logits, concepts, attns
        gc.collect(); torch.cuda.empty_cache()
        continue

    # prepare common x for interpolation
    def maybe_normalize(arr):
        if arr.size == 0: return arr
        if not normalize_speed: return arr
        mn, mx = arr.min(), arr.max()
        if mx - mn < 1e-9: return np.zeros_like(arr)
        return (arr - mn) / (mx - mn + 1e-12)

    speed_graph_norm = maybe_normalize(speed_graph)
    L = len(speed_graph_norm) if speed_graph_norm.size > 0 else 1
    x_global = np.arange(L)

    # find peaks in the attention signal for highlighting
    peaks = []
    if speed_graph_norm.size > 0 and len(speed_graph_norm) > 1:
        peaks, _ = find_peaks(speed_graph_norm, distance=max(4, len(speed_graph_norm)//20))

    print(f"[BATCH {j}] Found {len(peaks)} attention peaks at positions: {peaks.tolist()}")

    # -------- chunked plotting & safe saving --------
    for chunk_start in range(0, len(frames_indices), chunk_size):
        chunk = frames_indices[chunk_start: chunk_start + chunk_size]
        ncol = len(chunk)
        if ncol == 0:
            continue

        fig, axes = plt.subplots(2, ncol, figsize=(fig_w_per_col * ncol, fig_h),
                                 gridspec_kw={'height_ratios':[3,1]})
        axes = np.array(axes)
        axes_flat = axes.flatten()

        for k, frame_idx in enumerate(chunk):
            ax_img = axes_flat[k]
            ax_plot = axes_flat[k + ncol]

            # image from CPU tensor (C,H,W) -> permute to H,W,C
            img_tensor = img_cpu[0][frame_idx]
            image_frame = img_tensor.permute(1,2,0).numpy()

            # image pane
            ax_img.imshow((image_frame * 255).astype(np.uint8))
            title = '\n'.join([split_string("• " + h) for h in (f[frame_idx - 10] if 0 <= frame_idx-10 < len(f) else [])])
            ax_img.set_title(title, fontsize=9)
            ax_img.axis("off")

            # plot attention signal (single line, solid style)
            plotted_any = False
            if speed_graph_norm.size > 0:
                # single solid line for the last layer attention
                ax_plot.plot(x_global, speed_graph_norm, 
                           label='Attention', 
                           color="#1f77b4", 
                           linestyle="-",      # solid line
                           linewidth=2.5, 
                           alpha=0.9)
                plotted_any = True

                # mark peaks (global)
                for p in peaks:
                    if 0 <= p < len(speed_graph_norm):
                        ax_plot.plot(p, speed_graph_norm[p], marker='v', markersize=6, 
                                   color='darkred', alpha=0.9, markeredgecolor='white', linewidth=1)

                # highlight current position and put marker with zoom around
                pos = frame_idx - 10
                # adaptation between frame index and x_global scale:
                # if pos exceeds x_global limits, clip it
                pos_clipped = int(np.clip(pos, 0, len(x_global)-1))
                if 0 <= pos_clipped < len(x_global):
                    # marker for current position
                    y_val = float(speed_graph_norm[pos_clipped])
                    ax_plot.scatter(pos_clipped, y_val, color=marker_color, s=marker_size, zorder=5,
                                  edgecolor='white', linewidth=2)
                    if annotate_values:
                        # write value next to marker
                        ax_plot.annotate(f"{y_val:.3f}", xy=(pos_clipped, y_val), xytext=(5, 8),
                                         textcoords="offset points", fontsize=8, color=marker_color,
                                         fontweight='bold')
                    '''
                    # automatic zoom around current frame
                    x_min = max(0, pos_clipped - zoom_window)
                    x_max = min(len(x_global)-1, pos_clipped + zoom_window)
                    
                    # extend y margin based on zoomed portion
                    y_vals_chunk = speed_graph_norm[x_min:x_max+1]

                    
                    if len(y_vals_chunk) > 0:
                        y_min_z = min(y_vals_chunk); y_max_z = max(y_vals_chunk)
                        ypad = max(1e-6, (y_max_z - y_min_z) * 0.15)
                        ax_plot.set_ylim(y_min_z - ypad, y_max_z + ypad)
                    ax_plot.set_xlim(x_min, x_max)
                    '''
                    # no zoom: plot full sequence [0,240]
                    ax_plot.set_xlim(0, 240)
                    ax_plot.set_ylim(speed_graph_norm.min() - 0.05, 
                                     speed_graph_norm.max() + 0.05)

                    # add vertical line for current position
                    ax_plot.axvline(x=pos_clipped, color=marker_color, linestyle='--', 
                                  alpha=0.6, linewidth=1.5)

            # highlight intervention bands if mask is compatible
            if (intervention_mask is not None) and (len(intervention_mask) == img_cpu.shape[1]):
                seg_on=False; start=None
                for t_i, val in enumerate(intervention_mask):
                    if val and not seg_on: seg_on=True; start=t_i
                    if (not val) and seg_on: 
                        seg_on=False; end=t_i
                        ax_plot.add_patch(patches.Rectangle((start, ax_plot.get_ylim()[0]), 
                                                          end-start, 
                                                          ax_plot.get_ylim()[1]-ax_plot.get_ylim()[0], 
                                                          color='magenta', alpha=0.06, linewidth=0))
                if seg_on:
                    end = len(intervention_mask)-1
                    ax_plot.add_patch(patches.Rectangle((start, ax_plot.get_ylim()[0]), 
                                                      end-start, 
                                                      ax_plot.get_ylim()[1]-ax_plot.get_ylim()[0], 
                                                      color='magenta', alpha=0.06, linewidth=0))


            # legend / labels / grid
            if plotted_any:
                ax_plot.legend(fontsize=6, loc='upper right')
                # add info text box
                #info_text = f"Peaks: {len(peaks)}\nVideo: {current_video_id[:20]}..."
                #ax_plot.text(0.02, 0.98, info_text, transform=ax_plot.transAxes,
                 #          fontsize=7, verticalalignment='top',bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.7))
            else:
                ax_plot.text(0.5, 0.5, "No attention data", ha='center', va='center', fontsize=9, color='gray')
                ax_plot.set_ylim(0, 1)

            ax_plot.set_xlabel("Sequence position", fontsize=9)
            ax_plot.set_ylabel("Attention intensity", fontsize=9)
            ax_plot.tick_params(axis='both', which='major', labelsize=8)
            ax_plot.grid(axis='y', linestyle=':', linewidth=0.5, alpha=0.6)

        plt.tight_layout()
        
        # include video ID in filename for clarity
        video_id_short = current_video_id.replace('_', '-')[:30]  # shorten and sanitize
        png_path = os.path.join(outdir, f"{video_id_short}_batch{j}_chunk{chunk_start}.png")
        safe_plot_and_save(fig, png_path, dpi=dpi_save)
        saved_pngs.append(png_path)

    # create GIF (if requested)
    if make_gif and saved_pngs:
        try:
            frames = [Image.open(p) for p in saved_pngs]
            video_id_short = current_video_id.replace('_', '-')[:30]
            gif_path = os.path.join(outdir, f"{video_id_short}_attention_batch{j}.gif")
            frames[0].save(gif_path, format='GIF', append_images=frames[1:], save_all=True, duration=600, loop=0)
            for im in frames: im.close()
            print(f"[INFO] GIF created: {gif_path}")
        except Exception as e:
            print(f"[WARN] GIF creation failed: {e}")

    # memory cleanup
    del img, angle_dev, distance_dev, vego_dev, logits, concepts, attns
    gc.collect()
    torch.cuda.empty_cache()

    # optional: save checkpoint and free model
    if save_model_checkpoint:
        try:
            torch.save(model.state_dict(), checkpoint_path)
            print(f"[INFO] model saved to {checkpoint_path}")
        except Exception as e:
            print(f"[WARN] save model failed: {e}")

    # show list of generated files and first image for verification (if exists)
    print(f"[DONE] Video {current_video_id} - saved {len(saved_pngs)} images")
    if saved_pngs:
        print(f"[INFO] Files saved: {saved_pngs[:3]}...")  # show first 3
        try:
            from IPython.display import display
            display(Image.open(saved_pngs[0]))
        except:
            print("[INFO] Display not available in this environment")

    # if processing specific video, break after finding it
    if not PROCESS_ALL_VIDEOS and TARGET_VIDEO in current_video_id:
        print(f"[COMPLETED] Finished processing target video: {TARGET_VIDEO}")
        break
    
    if j >= 10:  # safety limit - process max 10 batches
        break

print("[COMPLETED] Single layer attention visualization with video selection")